# fightcal free GPU runner
Enable a T4 (preferred) or P100 GPU and internet. Attach mdabbert's Ultimate UFC Dataset.
Budget estimates: base inference 20-90 minutes; SFT 30-90 minutes plus inference;
each 300-step GRPO arm 2-6 hours plus inference. Hardware and output length matter.
Run one arm per session using RUN_ARMS. Nothing needs to run locally.
On Colab select Runtime > Change runtime type > T4 GPU, then upload ufc-master.csv.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path('/kaggle/working' if Path('/kaggle').exists() else '/content')
REPO = ROOT / 'fight-calibration-rl'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/mengtobby/fight-calibration-rl', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[model]'], check=True)
RUNS = ROOT / 'runs'
RUNS.mkdir(exist_ok=True)
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], check=True)


In [ ]:
candidates = sorted(Path('/kaggle/input').rglob('ufc-master.csv')) if Path('/kaggle/input').exists() else sorted(Path('/content').glob('ufc-master.csv'))
if not candidates:
    raise FileNotFoundError('Attach the Ultimate UFC Dataset on Kaggle, or upload ufc-master.csv to /content on Colab, then rerun this cell.')
run('scripts/prepare_data.py', '--csv', candidates[0], '--require-odds')
run('scripts/run_baselines.py', '--train', 'data/processed/train.jsonl', '--test', 'data/processed/test.jsonl', '--out', RUNS / 'baselines.jsonl')


Select arms below. Save/download runs before ending a session. Training writes adapters and checkpoints under runs/<arm>; prediction JSONL is appended batch by batch. Re-running prediction with --resume skips completed bout/variant pairs. Training checkpoint resume is not exposed by the current CLI; restart interrupted training in a fresh output directory and preserve finished adapters. For GPU memory errors reduce training batch size while keeping GRPO batch size divisible by num_generations, and reduce prediction batch size.


In [ ]:
import json
RUN_ARMS = ['base']  # Add sft_outcome, grpo_correct, grpo_brier, grpo_log, grpo_brier_odds.
MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
for arm in RUN_ARMS:
    adapter = []
    if arm != 'base':
        cfg = json.loads(Path(f'configs/{arm}.json').read_text())
        cfg['output_dir'] = str(RUNS / arm)
        config_path = RUNS / f'{arm}.json'
        config_path.write_text(json.dumps(cfg, indent=2))
        script = 'scripts/train_sft.py' if arm == 'sft_outcome' else 'scripts/train_grpo.py'
        run(script, '--config', config_path)
        adapter = ['--adapter', cfg['output_dir']]
    mode = 'pick' if arm == 'sft_outcome' else 'reason_prob'
    variants = 'base,swapped' if mode == 'pick' else 'base,swapped,reasoning_swap'
    if 'odds' in arm:
        variants += ',odds_in_prompt,odds_perturbed'
    run('scripts/predict.py', '--arm', arm, '--model', MODEL, *adapter,
        *(['--include-odds'] if 'odds' in arm else []),
        '--mode', mode, '--bouts', 'data/processed/test.jsonl', '--variants', variants,
        '--batch-size', '2', '--out', RUNS / f'{arm}.jsonl', '--resume')


In [ ]:
predictions = sorted(RUNS.glob('*.jsonl'))
run('scripts/report.py', '--predictions', *predictions, '--out-dir', RUNS / 'report')
from IPython.display import Markdown, display
display(Markdown((RUNS / 'report/report.md').read_text()))
